In [7]:
import os
import re
import pandas as pd

# Step A: Identify Sample_id and tool

In [8]:



FILENAME_PATTERN = re.compile(
    r"^(?P<sample_id>.+?)_(?P<tool>fusioncatcher|arriba)_case_specific_filtered\.tsv$"
)


def parse_filename(filename):

    match = FILENAME_PATTERN.match(filename)

    if match is None:
        return None, None

    return match.group("sample_id"), match.group("tool")


# Step B: ARRIBA fusion_transcript Cleaner

In [9]:
def clean_arriba_fusion_transcript(fusion_transcript):

    if fusion_transcript is None or fusion_transcript.strip() in ("", "."):
        return None, None, None, False

    sequence = fusion_transcript.strip()

    truncated = False

    if "..." in sequence:
        sequence = sequence.split("...")[0]
        truncated = True

    parts = sequence.split("|")

    if len(parts) == 2:
        left_raw, right_raw = parts
        insertion_raw = ""

    elif len(parts) == 3:
        left_raw, insertion_raw, right_raw = parts

    else:
        
        return None, None, None, truncated

    def clean_segment(segment):
        segment = segment.replace("___", "")     
        segment = segment.replace("-", "")       
        segment = segment.replace("[", "").replace("]", "") 
        segment = segment.replace("?", "N")    
        segment = segment.upper()
        return segment

    left_fragment = clean_segment(left_raw)
    right_fragment = clean_segment(right_raw)
    insertion_sequence = clean_segment(insertion_raw)

    return left_fragment, right_fragment, insertion_sequence, truncated


# Step C: Raw adapters per tool

In [10]:

def parse_fusioncatcher_row(row, sample_id):

    dna_fs = row.get("Fusion_sequence", "")

    return {

        "Sample_ID": sample_id,
        "Tool": "fusioncatcher",
        "Gene_A_name": row.get("Gene_1_symbol(5end_fusion_partner)", ""),
        "Gene_B_name": row.get("Gene_2_symbol(3end_fusion_partner)", ""),
        "Gene_A_ensembl_id": row.get("Gene_1_id(5end_fusion_partner)", ""),
        "Gene_B_ensembl_id": row.get("Gene_2_id(3end_fusion_partner)", ""),
        "dna_fs": dna_fs,
        "Insertion_sequence": "",
        "reported_frame": row.get("Predicted_effect", ""),
        "Spanning_unique_reads": row.get("Spanning_unique_reads", ""),
        "Spanning_pairs": row.get("Spanning_pairs", ""),
        "Truncated_due_to_low_coverage": False,
        "Parse_successful": dna_fs not in ("", None) and "*" in dna_fs,

    }


def parse_arriba_row(row, sample_id):

    fusion_transcript = row.get("fusion_transcript", "")

    left_fragment, right_fragment, insertion_sequence, truncated = \
        clean_arriba_fusion_transcript(fusion_transcript)

    parse_successful = left_fragment is not None and right_fragment is not None

    if parse_successful:
        dna_fs = left_fragment + "*" + right_fragment
    else:
        dna_fs = ""

    return {

        "Sample_ID": sample_id,
        "Tool": "arriba",
        "Gene_A_name": row.get("gene1", ""),
        "Gene_B_name": row.get("gene2", ""),
        "Gene_A_ensembl_id": row.get("gene_id1", ""),
        "Gene_B_ensembl_id": row.get("gene_id2", ""),
        "dna_fs": dna_fs,
        "Insertion_sequence": insertion_sequence if insertion_sequence else "",
        "reported_frame": row.get("reading_frame", ""),
        "Confidence": row.get("confidence", ""),
        "Arriba_peptide_sequence": row.get("peptide_sequence", ""),
        "Split_reads1": row.get("split_reads1", ""),
        "Split_reads2": row.get("split_reads2", ""),
        "Truncated_due_to_low_coverage": truncated,
        "Parse_successful": parse_successful,

    }



# Step D: Scan the sample files folder and build a master table

In [11]:
def load_all_sample_fusions(folder_path):

    all_fusion_records = []

    for filename in sorted(os.listdir(folder_path)):

        sample_id, tool = parse_filename(filename)

        if sample_id is None:
            continue   # not a recognized fusion file, skip it

        file_path = os.path.join(folder_path, filename)

        df = pd.read_csv(file_path, sep="\t")

      
        df.columns = [column.lstrip("#") for column in df.columns]

        for _, row in df.iterrows():

            if tool == "fusioncatcher":
                record = parse_fusioncatcher_row(row, sample_id)
            else:
                record = parse_arriba_row(row, sample_id)

            all_fusion_records.append(record)

    master_fusion_table = pd.DataFrame(all_fusion_records)

    return master_fusion_table


# Run

if __name__ == "__main__":

    folder_path = input("Enter path to the folder containing per-sample fusion TSVs: ")

    master_fusion_table = load_all_sample_fusions(folder_path)

    print("\nTotal fusion records loaded:", len(master_fusion_table))
    print("\nBy tool:")
    print(master_fusion_table["Tool"].value_counts())
    print("\nBy sample:")
    print(master_fusion_table["Sample_ID"].value_counts())

    print("\nParse failures:")
    print(master_fusion_table[~master_fusion_table["Parse_successful"]][["Sample_ID", "Tool"]].value_counts())

    master_fusion_table.to_csv("master_fusion_table.csv", index=False)
    print("\nSaved as master_fusion_table.csv")

Enter path to the folder containing per-sample fusion TSVs:  /home/jerrybryt/MSBT/IP/Gene_Fusions/Case_specific_fusions/Case_specific_fusions



Total fusion records loaded: 1975

By tool:
Tool
fusioncatcher    1408
arriba            567
Name: count, dtype: int64

By sample:
Sample_ID
AKU076     179
KAIC016    176
KAIC029    142
AKU066     141
AKU082     131
KAIC023    130
KAIC027    105
KAIC010     93
KAIC018     84
KAIC009     83
KAIC026     78
AKU040      77
AKU085      68
KAIC030     66
AKU073      63
AKU089      63
KAIC012     58
AKU041      55
AKU087      48
KAIC017     46
KAIC014     33
KAIC032     28
KAIC033     28
Name: count, dtype: int64

Parse failures:
Sample_ID  Tool  
KAIC029    arriba    15
KAIC010    arriba    11
KAIC012    arriba    11
KAIC023    arriba    11
AKU082     arriba     9
AKU076     arriba     8
KAIC016    arriba     8
AKU066     arriba     7
KAIC027    arriba     6
AKU041     arriba     4
KAIC009    arriba     4
AKU073     arriba     3
AKU089     arriba     3
KAIC018    arriba     3
KAIC026    arriba     3
AKU085     arriba     2
KAIC014    arriba     2
KAIC030    arriba     2
KAIC032    arriba   